In [1]:
%load_ext autoreload
%autoreload 2


import numpy as np
import jax.numpy as jnp

from gkp_optimal_control.states import gkp_states

from gkp_optimal_control.gate_results import (
    save_gate_result, load_gate_result, gate_result_path,
)
from gkp_optimal_control.gate_optimization import (
    optimize_gate_sequence, sequence_history,
)
from gkp_optimal_control.utils import to_ket
from gkp_optimal_control import pulses, pulse_simulation as ps

In [2]:
params, extras = pulses.load_config("../devices/device_eickbusch.toml")
N = extras["ecd"]["n_fock"]

In [3]:
N = 100
gkp_delta = 0.3
gkp_cutoff = 10
gkp_alpha = jnp.sqrt(jnp.pi / 2)
gkp_beta = jnp.sqrt(jnp.pi / 2) * 1j

energy_bound = 1.0

gkp_0, gkp_1 = gkp_states(N, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)
psi_target = gkp_0
psi_target = np.asarray(to_ket(psi_target)).reshape(-1)

In [7]:
path = "../data/gate_optimization/gkp_code_gate_params/ecd_g10.json"
result = load_gate_result(path)
traj = result.repropagate()            # cheap propagation from stored params

F_lin = ps.check_compile_convention(
    result, params, psi_target, N,
    alpha0=extras["ecd"]["alpha0"], tol=extras["ecd"]["tol"], frame_sign=-1.0,
)
seq = pulses.compile_from_result(
    result, params=params.replace(kappa=0.0),
    alpha0=extras["ecd"]["alpha0"], tol=extras["ecd"]["tol"], frame_sign=-1.0,
)
T = seq.duration
res = ps.simulate_sequence(seq, n_fock=N, save_every=1)
print(f"T = {T*1e3:.0f} ns, truncation = {res.truncation_error():.1e}")

T = 4108 ns, truncation = 5.2e-16


In [8]:
# joint fidelity: displace the target into the frame, compare to the |g> block, no renormalization
psi_t = ps._displace_np(N, -res.alpha[-1]) @ psi_target
F0 = abs(np.vdot(psi_t, res.blocks()[-1, 0])) ** 2

# first-order loss integral, frame-invariant, includes which-path dephasing
blk = res.blocks()
a_c = np.diag(np.sqrt(np.arange(1, N)), 1)
a_exp = np.einsum("tsi,ij,tsj->t", blk.conj(), a_c, blk)
var = res.photon_number(lab=False) - np.abs(a_exp) ** 2
I_ecd = np.trapezoid(var, res.times)

# brachistochrone at the same T
overlap = abs(psi_target[0])
theta = np.arccos(overlap)
n_perp = np.sum(np.arange(N) * abs(psi_target) ** 2) / np.sin(theta) ** 2
I_brach = T * n_perp * (0.5 - np.sin(2 * theta) / (4 * theta))

print(f"F(0) = {F0:.5f}   1-F estimate at device kappa: "
      f"ECD {params.kappa*I_ecd:.2e}, brachistochrone {params.kappa*I_brach:.2e}")

F(0) = 0.96734   1-F estimate at device kappa: ECD 2.77e-02, brachistochrone 1.81e-02


In [9]:
print(f"ideal F = {result.fidelity:.5f}, compiled F(0) = {F0:.5f}")

ideal F = 0.99095, compiled F(0) = 0.96734


In [11]:
m = seq.meta
print(np.abs(m["betas_realized"] - m["betas_target"]))   # should be ~tol * |beta|
print(np.abs(m["residual_lambda"]), m["alpha0_used"])

[2.83328929e-07 7.04135027e-07 1.14837982e-06 1.92316734e-06
 2.78838815e-06 1.96592179e-07 4.73073394e-07 6.77127216e-08
 1.61553060e-04 5.70030243e-07]
[3.98287986e-05 1.71139073e-05 5.31358720e-06 8.01888580e-06
 2.98673096e-06 3.50045925e-05 9.04938502e-06 4.22321914e-05
 4.78868791e-05 1.30514505e-07] [29.94229183 29.89231467 29.81849605 29.7890989  29.64578028 29.95068503
 29.89873534 29.97207027 29.98739545 11.90391673]


In [10]:
for label, pp in [("chi' only", params.replace(kappa=0.0, kerr=0.0)),
                  ("Kerr only", params.replace(kappa=0.0, chi_prime=0.0)),
                  ("both",      params.replace(kappa=0.0))]:
    s_ = pulses.compile_from_result(result, params=pp, alpha0=extras["ecd"]["alpha0"],
                                    tol=extras["ecd"]["tol"], frame_sign=-1.0)
    r_ = ps.simulate_sequence(s_, n_fock=N, save_every=10**9)
    pt_ = ps._displace_np(N, -r_.alpha[-1]) @ psi_target
    print(f"{label:<10} F(0) = {abs(np.vdot(pt_, r_.blocks()[-1, 0]))**2:.5f}")

chi' only  F(0) = 0.98132
Kerr only  F(0) = 0.98534
both       F(0) = 0.96734


In [25]:
import jax.numpy as jnp
from gkp_optimal_control.gate_optimization import build_sequence

def ideal_fidelity(n):
    gs = build_sequence("ecd", result.n_gates, n)
    vac = jnp.zeros((1, n), jnp.complex128).at[0, 0].set(1.0)
    psi = gs.propagate(jnp.asarray(result.flat_params), gs.lift(vac))[0]
    g0, _ = gkp_states(n, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)
    tgt = jnp.asarray(to_ket(g0)).reshape(-1)
    return float(abs(jnp.vdot(tgt, psi[0, 0])) ** 2)

for n in (40, 60, 80, 100, 120):
    print(n, ideal_fidelity(n))

40 0.9917666014748552
60 0.9909784536034301
80 0.9909513834521723
100 0.9909510010703043
120 0.9909509824017821


In [24]:
n = 100
gs = build_sequence("ecd", result.n_gates, n)
vac = jnp.zeros((1, n), jnp.complex128).at[0, 0].set(1.0)
out = gs.propagate(jnp.asarray(result.flat_params), gs.lift(vac))
print(len(out), [getattr(x, "shape", x) for x in out])

4 [(1, 2, 100), (), (10,), None]


In [28]:
s = seq_lin                                    # linear compile (chi' = K = 0, kappa = 0)
r = ps.simulate_sequence(s, n_fock=N, save_every=1)
blk = r.blocks()                               # (n_samples+1, 2, N); index = sample index

gs = build_sequence("ecd", result.n_gates, N)
vac = jnp.zeros((1, N), jnp.complex128).at[0, 0].set(1.0)
hist = np.asarray(gs.history(jnp.asarray(result.flat_params), gs.lift(vac)))[:, 0]
print(hist.shape, len(s.gate_indices))         # expect (2*10+2, 2, N) and 21

idx = np.concatenate([[0], s.gate_indices])
labels = ["input"] + [seg.label for seg in s.segments]
for k, i in enumerate(idx):
    D = ps._displace_np(N, -r.alpha_full[i])   # move the ideal state into the frame
    ideal = hist[k] @ D.T                      # displace both qubit blocks
    ov_g = abs(np.vdot(ideal[0], blk[i, 0]))
    ov_e = abs(np.vdot(ideal[1], blk[i, 1]))
    print(f"{k:2d} {labels[k]:<28} F = {(ov_g + ov_e) ** 2:.5f}")

(22, 2, 100) 21
 0 input                        F = 1.00000
 1 R(+0.500pi, -0.503pi)        F = 0.99991
 2 ECD(+2.454-0.737j)           F = 1.00000
 3 R(+0.499pi, +1.024pi)        F = 0.99502
 4 ECD(-0.127-1.927j)           F = 0.99443
 5 R(+0.309pi, -1.012pi)        F = 0.99322
 6 ECD(-0.626-1.079j)           F = 0.99335
 7 R(+0.256pi, +0.017pi)        F = 0.98862
 8 ECD(-1.313+0.665j)           F = 0.99309
 9 R(+0.145pi, -0.011pi)        F = 0.99202
10 ECD(-0.367-0.832j)           F = 0.99191
11 R(+0.666pi, +1.014pi)        F = 0.98784
12 ECD(-2.136+1.213j)           F = 0.99216
13 R(+0.460pi, +0.005pi)        F = 0.98312
14 ECD(-0.011+1.545j)           F = 0.98533
15 R(+0.589pi, -0.998pi)        F = 0.99071
16 ECD(+2.267-1.284j)           F = 0.98960
17 R(+1.189pi, +1.019pi)        F = 0.97786
18 ECD(-2.500-1.057j)           F = 0.98116
19 R(-0.484pi, -0.996pi)        F = 0.93912
20 ECD(-0.190+0.106j)           F = 0.94047
21 R(+0.500pi, +0.496pi)        F = 0.94102


In [27]:
for fs in (+1.0, -1.0):
    s = pulses.compile_from_result(result, params=p_lin, alpha0=extras["ecd"]["alpha0"],
                                   tol=extras["ecd"]["tol"], frame_sign=fs)
    r = ps.simulate_sequence(s, n_fock=N, save_every=1)
    pt = ps._displace_np(N, -r.alpha[-1]) @ psi_target
    print(fs, abs(np.vdot(pt, r.blocks()[-1, 0])) ** 2)

1.0 0.9325351651832197
-1.0 0.9893104880631666


In [29]:
rot_frames = [seg.meta["frame"] for seg in s.segments if seg.kind == "rotation"]
rows = [k for k, seg in enumerate(s.segments, start=1) if seg.kind == "rotation"]

for j, k in enumerate(rows):
    i = idx[k - 1]                              # boundary just BEFORE rotation j
    D = ps._displace_np(N, -r.alpha_full[i])
    ideal = hist[k - 1] @ D.T
    ov_g = np.vdot(ideal[0], blk[i, 0])
    ov_e = np.vdot(ideal[1], blk[i, 1])
    measured = np.angle(ov_g) - np.angle(ov_e)
    wrap = lambda x: (x + np.pi) % (2 * np.pi) - np.pi
    print(f"rot {j:2d}  measured {wrap(measured):+.4f}  compiler F {rot_frames[j]:+.4f}  "
          f"diff(+) {wrap(measured - rot_frames[j]):+.4f}  diff(-) {wrap(measured + rot_frames[j]):+.4f}")

rot  0  measured +0.0000  compiler F +0.0000  diff(+) +0.0000  diff(-) +0.0000
rot  1  measured +0.0896  compiler F -0.0656  diff(+) +0.1552  diff(-) +0.0239
rot  2  measured +0.1247  compiler F +0.0293  diff(+) +0.0954  diff(-) +0.1539
rot  3  measured -0.0441  compiler F -0.0457  diff(+) +0.0016  diff(-) -0.0898
rot  4  measured +0.0831  compiler F +0.0239  diff(+) +0.0592  diff(-) +0.1070
rot  5  measured -0.0156  compiler F -0.0338  diff(+) +0.0182  diff(-) -0.0494
rot  6  measured +0.1439  compiler F -0.0261  diff(+) +0.1699  diff(-) +0.1178
rot  7  measured +0.0516  compiler F +0.0022  diff(+) +0.0493  diff(-) +0.0538
rot  8  measured +0.2313  compiler F -0.0703  diff(+) +0.3016  diff(-) +0.1609
rot  9  measured +0.4119  compiler F -0.0042  diff(+) +0.4161  diff(-) +0.4077
rot 10  measured +0.0147  compiler F +0.0035  diff(+) +0.0112  diff(-) +0.0182


In [30]:
import dataclasses

def calibrate_theta_primes(seq, n_fock):
    """Measure the qubit phase each compiled ECD pulse actually imparts, one gate at a time."""
    p, vac = seq.params, np.eye(n_fock)[0]
    psi0 = np.zeros((2, n_fock), complex)
    psi0[0, 0] = psi0[1, 0] = 1 / np.sqrt(2)               # |0>|+>
    cal = []
    for seg in seq.segments:
        if seg.kind != "ecd":
            continue
        sl, L = seg.slice(), seg.stop - seg.start
        sub = pulses.PulseSequence(
            eps=seq.eps[sl], omega=seq.omega[sl], params=p, delta=seq.delta,
            segments=[pulses.PulseSegment("ecd", 0, L, seg.label)],
            gate_indices=np.array([L]), meta={},
        )
        r1 = ps.simulate_sequence(sub, psi0=psi0, n_fock=n_fock, save_every=10**9)
        b, aT, beta = r1.blocks()[-1], r1.alpha_full[-1], seg.meta["beta"]
        Dm = ps._displace_np(n_fock, -aT)
        ideal_g = Dm @ ps._displace_np(n_fock, -beta / 2) @ vac / np.sqrt(2)  # from |e> branch
        ideal_e = Dm @ ps._displace_np(n_fock, +beta / 2) @ vac / np.sqrt(2)  # from |g> branch
        og, oe = np.vdot(ideal_g, b[0]), np.vdot(ideal_e, b[1])
        cal.append(-(np.angle(og) - np.angle(oe)))           # measured = -theta' after the echo
        print(f"{seg.label:<20} |og|+|oe| = {abs(og) + abs(oe):.6f}   "
              f"theta' compiler {seg.meta['theta_prime']:+.4f}  calibrated {cal[-1]:+.4f}")
    return np.array(cal)

def apply_frames(seq, theta_primes, frame_sign=1.0):
    """Rebuild rotation pulses with the virtual-Z update F -> -F + theta'."""
    omega, F, j = seq.omega.copy(), 0.0, 0
    for seg in seq.segments:
        if seg.kind == "rotation":
            omega[seg.slice()] = pulses.rotation_waveform(
                seg.meta["theta"], seg.meta["phi_ideal"] - F, seq.params)
        elif seg.kind == "ecd":
            F = -F + frame_sign * theta_primes[j]
            j += 1
    return dataclasses.replace(seq, omega=omega)

In [31]:
# 1. sanity: the compiler's own theta' must reproduce the original waveform exactly
assert np.allclose(apply_frames(s, s.meta["theta_primes"]).omega, s.omega)

# 2. calibrate and rebuild
th_cal = calibrate_theta_primes(s, N)
s_cal = apply_frames(s, th_cal)

# 3. joint fidelity
r = ps.simulate_sequence(s_cal, n_fock=N, save_every=1)
pt = ps._displace_np(N, -r.alpha[-1]) @ psi_target
print(abs(np.vdot(pt, r.blocks()[-1, 0])) ** 2)

ECD(+2.454-0.737j)   |og|+|oe| = 1.000000   theta' compiler -0.0656  calibrated -0.0417
ECD(-0.127-1.927j)   |og|+|oe| = 1.000000   theta' compiler -0.0364  calibrated -0.0184
ECD(-0.626-1.079j)   |og|+|oe| = 1.000000   theta' compiler -0.0165  calibrated -0.0048
ECD(-1.313+0.665j)   |og|+|oe| = 1.000000   theta' compiler -0.0218  calibrated -0.0081
ECD(-0.367-0.832j)   |og|+|oe| = 1.000000   theta' compiler -0.0099  calibrated -0.0014
ECD(-2.136+1.213j)   |og|+|oe| = 1.000000   theta' compiler -0.0599  calibrated -0.0369
ECD(-0.011+1.545j)   |og|+|oe| = 1.000000   theta' compiler -0.0238  calibrated -0.0094
ECD(+2.267-1.284j)   |og|+|oe| = 1.000000   theta' compiler -0.0681  calibrated -0.0438
ECD(-2.500-1.057j)   |og|+|oe| = 1.000000   theta' compiler -0.0746  calibrated -0.0492
ECD(-0.190+0.106j)   |og|+|oe| = 1.000000   theta' compiler -0.0007  calibrated +0.0001
0.948646927608149


In [11]:
alpha = seq.frame_trajectory()
c = 4 * params.kerr + 2 * params.chi_prime
phi = np.array([c * np.sum(np.abs(alpha[g.start:g.stop]) ** 2) * params.dt
                for g in seq.segments if g.kind == "ecd"])
Phi = np.cumsum(phi)
pre = np.concatenate([[0.0], Phi[:-1]])          # rotation accumulated before gate i
print("phi per gate (mrad):", np.round(1e3 * phi, 2), "  total:", Phi[-1])

p = result.params
betas = np.asarray(p["betas"])[: phi.size]
for sgn in (+1, -1):
    s_ = pulses.compile_ecd_sequence(
        betas * np.exp(1j * sgn * pre), p["thetas"], p["phis"],
        alpha0=extras["ecd"]["alpha0"], params=params.replace(kappa=0.0),
        tol=extras["ecd"]["tol"], frame_sign=-1.0,
    )
    r_ = ps.simulate_sequence(s_, n_fock=N, save_every=10**9)
    tgt = np.exp(1j * sgn * Phi[-1] * np.arange(N)) * psi_target   # target in the rotated cavity frame
    pt_ = ps._displace_np(N, -r_.alpha[-1]) @ tgt
    print(f"sign {sgn:+d}: F(0) = {abs(np.vdot(pt_, r_.blocks()[-1, 0]))**2:.5f}")

phi per gate (mrad): [10.32  7.65  4.76  5.7   3.32  9.88  6.02 10.51 10.98  0.3 ]   total: 0.06944953369745759
sign +1: F(0) = 0.98544
sign -1: F(0) = 0.88893
